# Model Compression

This notebook compares INT8 quantization, structured pruning, and knowledge distillation for the intrusion detection model.

The smaller neural encoders are trained using UNSW data. Each model is then adapted using the **same 1,000 CIC labels** and evaluated on the **same CIC development records**. The comparison includes detection performance, model storage, and CPU inference time.

The data samples, feature schema, and fixed label selection are prepared in notebooks `01_data_preparation.ipynb` and `02_source_model_and_adaptation.ipynb`.

In [ ]:
from pathlib import Path
import copy
import json
import time
import warnings
from IPython.display import display

import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import RobustScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, average_precision_score

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Local environment: set BASE to the project data folder.')

BASE = Path('/content/drive/MyDrive/datanew/a4_analysis')
ADAPT_DIR = BASE / 'reproduction_runs' / 'source_adaptation'
OUT_DIR = BASE / 'reproduction_runs' / 'model_compression'
OUT_DIR.mkdir(parents=True, exist_ok=True)

SOURCE_CSV = BASE / 'UNSW_100k.csv'
TARGET_CSV = BASE / 'CIC_100k.csv'
SCHEMA_JSON = BASE / 'csc786_feature_schema.json'
CONFIG_JSON = ADAPT_DIR / 'run_config.json'
SOURCE_SCALER = ADAPT_DIR / 'source_standard_scaler.joblib'
SELECTED_INDICES = ADAPT_DIR / 'selected_indices_seed51.npy'

for path in (SOURCE_CSV, TARGET_CSV, SCHEMA_JSON, CONFIG_JSON,
             SOURCE_SCALER, SELECTED_INDICES):
    if not path.is_file():
        raise FileNotFoundError(f'Missing experiment input: {path}')

SEED = 44001
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(2)
device = torch.device('cpu')

with SCHEMA_JSON.open(encoding='utf-8') as f:
    schema = json.load(f)
with CONFIG_JSON.open(encoding='utf-8') as f:
    run_config = json.load(f)

feature_names = schema['final_features_in_order']
raw_features = schema['raw_features']
assert len(feature_names) == 43
assert len(raw_features) == 41

checkpoint_name = run_config['source_checkpoint']
possible_checkpoints = [BASE / checkpoint_name, ADAPT_DIR / checkpoint_name]
checkpoint_path = next((p for p in possible_checkpoints if p.is_file()), None)
if checkpoint_path is None:
    raise FileNotFoundError(f'Source encoder checkpoint not found: {checkpoint_name}')

print('Source checkpoint:', checkpoint_path.name)
print('Output folder:', OUT_DIR)

## 1. Prepare the model inputs

The same 43 features and source-fitted scaler are used for all models. The CIC development set is kept separate from the adaptation pool, and repeated feature records are filtered from the development set.

In [ ]:
def add_flow_features(frame):
    frame = frame.copy()
    in_b = frame['IN_BYTES'].clip(lower=0)
    out_b = frame['OUT_BYTES'].clip(lower=0)
    in_p = frame['IN_PKTS'].clip(lower=0)
    out_p = frame['OUT_PKTS'].clip(lower=0)
    total_b = in_b + out_b
    total_p = in_p + out_p

    frame['FE_LOG_TOTAL_BYTES'] = np.log1p(total_b)
    frame['FE_LOG_TOTAL_PKTS'] = np.log1p(total_p)
    frame['FE_BYTE_ASYMMETRY'] = (in_b - out_b) / (total_b + 1)
    frame['FE_PKT_ASYMMETRY'] = (in_p - out_p) / (total_p + 1)
    frame['FE_LOG_AVG_PKT_SIZE'] = np.log1p(total_b / (total_p + 1))
    frame['FE_LOG_DURATION'] = np.log1p(
        frame['FLOW_DURATION_MILLISECONDS'].clip(lower=0)
    )
    return frame

source = add_flow_features(pd.read_csv(SOURCE_CSV))
target = add_flow_features(pd.read_csv(TARGET_CSV))
source_scaler = joblib.load(SOURCE_SCALER)
special_index = feature_names.index('SRC_TO_DST_SECOND_BYTES')


def transform_features(frame):
    x = frame[feature_names].to_numpy(dtype=np.float64, copy=True)
    x[:, special_index] = np.arcsinh(x[:, special_index])
    x = np.arcsinh(source_scaler.transform(x)).astype(np.float32)
    if not np.isfinite(x).all():
        raise ValueError('Non-finite model features')
    return x

# Same group splits as the source model and adaptation notebook.
source_groups = pd.util.hash_pandas_object(
    source[raw_features], index=False
).to_numpy()
train_idx, rest_idx = next(GroupShuffleSplit(
    n_splits=1, test_size=0.30, random_state=13
).split(source, groups=source_groups))
val_local, _ = next(GroupShuffleSplit(
    n_splits=1, test_size=0.50, random_state=13
).split(source.iloc[rest_idx], groups=source_groups[rest_idx]))
source_train = source.iloc[train_idx]
source_val = source.iloc[rest_idx[val_local]]

target_groups = pd.util.hash_pandas_object(
    target[raw_features], index=False
).to_numpy()
adapt_idx, dev_idx = next(GroupShuffleSplit(
    n_splits=1, test_size=0.50, random_state=786
).split(target, groups=target_groups))
target_adapt = target.iloc[adapt_idx]
target_dev = target.iloc[dev_idx]

pool_hash = pd.util.hash_pandas_object(
    target_adapt[feature_names], index=False
).to_numpy()
dev_hash = pd.util.hash_pandas_object(
    target_dev[feature_names], index=False
).to_numpy()
keep_dev = (
    ~pd.Series(dev_hash).duplicated(keep='first').to_numpy()
    & ~np.isin(dev_hash, np.unique(pool_hash))
)
target_dev = target_dev.iloc[np.flatnonzero(keep_dev)]

X_train = transform_features(source_train)
X_val = transform_features(source_val)
X_pool = transform_features(target_adapt)
X_dev = transform_features(target_dev)

SOURCE_CLASSES = [
    'Benign', 'Exploits', 'Fuzzers', 'Generic', 'Reconnaissance',
    'DoS', 'Backdoor', 'Analysis', 'Shellcode', 'Worms'
]
source_to_id = {name: i for i, name in enumerate(SOURCE_CLASSES)}


def source_label_ids(frame):
    binary = frame['Label'].to_numpy(dtype=int)
    attacks = frame['Attack'].fillna('Unknown_Attack').astype(str).str.strip().to_numpy()
    names = np.where(binary == 0, 'Benign', attacks)
    unexpected = set(names) - set(SOURCE_CLASSES)
    if unexpected:
        raise ValueError(f'Unexpected source classes: {unexpected}')
    return np.array([source_to_id[name] for name in names], dtype=np.int64)

y_train = source_label_ids(source_train)
y_val = source_label_ids(source_val)

selected = np.load(SELECTED_INDICES).astype(np.int64)
assert len(selected) == 1000 and len(np.unique(selected)) == 1000
assert selected.min() >= 0 and selected.max() < len(target_adapt)
assert len(X_pool) == len(target_adapt)

pool_binary = target_adapt['Label'].to_numpy(dtype=int)
pool_attack = target_adapt['Attack'].fillna('Unknown_Attack').astype(str).str.strip().to_numpy()
dev_binary = target_dev['Label'].to_numpy(dtype=int)
dev_attack = np.where(
    dev_binary == 0, 'Benign',
    target_dev['Attack'].fillna('Unknown_Attack').astype(str).str.strip().to_numpy()
)
dev_categories = sorted(set(dev_attack) - {'Benign'})

print('UNSW train / validation:', X_train.shape, X_val.shape)
print('CIC adaptation / development:', X_pool.shape, X_dev.shape)
print('Selected CIC labels:', len(selected))

## 2. Load the source model

The teacher encoder uses 64 hidden units and produces 32 neural features. The smaller models keep the 32-dimensional output but reduce the hidden layer to 32 or 16 units.

In [ ]:
class RareAwareNet(nn.Module):
    def __init__(self, input_dim=43, n_classes=10):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 64), nn.LayerNorm(64), nn.GELU(),
            nn.Linear(64, 32), nn.GELU()
        )
        self.decoder = nn.Sequential(
            nn.Linear(32, 64), nn.GELU(), nn.Linear(64, input_dim)
        )
        self.classifier = nn.Sequential(
            nn.LayerNorm(32), nn.Linear(32, n_classes)
        )

    def forward(self, x):
        z = self.encoder(x)
        return self.classifier(z), z


class CompactNIDS(nn.Module):
    def __init__(self, width, n_classes=10):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(43, width), nn.LayerNorm(width), nn.GELU(),
            nn.Linear(width, 32), nn.GELU()
        )
        self.classifier = nn.Sequential(
            nn.LayerNorm(32), nn.Linear(32, n_classes)
        )

    def forward(self, x):
        z = self.encoder(x)
        return self.classifier(z), z


checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=True)
if checkpoint.get('input_features') != feature_names:
    raise ValueError('Saved model and feature schema do not match')
if checkpoint.get('class_names') != SOURCE_CLASSES:
    raise ValueError('Saved source classes do not match')

teacher = RareAwareNet().cpu().eval()
teacher.load_state_dict(checkpoint['state_dict'])
for p in teacher.parameters():
    p.requires_grad_(False)


def encoder_parameters(model):
    return sum(p.numel() for p in model.encoder.parameters())


@torch.inference_mode()
def encode(model, features, batch_size=2048):
    model.eval()
    parts = []
    for start in range(0, len(features), batch_size):
        batch = torch.from_numpy(np.ascontiguousarray(features[start:start + batch_size]))
        parts.append(model.encoder(batch).cpu().numpy())
    return np.vstack(parts).astype(np.float32)


@torch.inference_mode()
def source_macro_f1(model):
    model.eval()
    preds = []
    for start in range(0, len(X_val), 2048):
        x = torch.from_numpy(np.ascontiguousarray(X_val[start:start + 2048]))
        preds.append(model(x)[0].argmax(1).cpu().numpy())
    return f1_score(y_val, np.concatenate(preds),
                    labels=np.arange(len(SOURCE_CLASSES)),
                    average='macro', zero_division=0)


source_counts = np.bincount(y_train, minlength=len(SOURCE_CLASSES))
source_weights = np.clip(
    (source_counts.max() / np.maximum(source_counts, 1)) ** 0.25,
    1, 8
)
source_weights_t = torch.tensor(source_weights, dtype=torch.float32)


def source_focal_loss(logits, labels):
    ce = F.cross_entropy(logits, labels, reduction='none')
    pt = F.softmax(logits, dim=1).gather(
        1, labels[:, None]
    ).squeeze(1).clamp(1e-7, 1)
    weights = source_weights_t[labels]
    return (weights * (1 - pt).pow(1.5) * ce).sum() / weights.sum()


print('Teacher encoder parameters:', encoder_parameters(teacher))
print('Teacher source validation macro-F1:', round(source_macro_f1(teacher), 5))

## 3. Structured pruning

Pruning removes hidden units with smaller outgoing weight magnitudes. The remaining weights are copied into smaller networks, then fine-tuned on UNSW training data. Source validation performance determines the saved checkpoint.

In [ ]:
def prune_encoder(model, width):
    compact = CompactNIDS(width, len(SOURCE_CLASSES))
    weights = model.encoder[3].weight.detach()
    importance = torch.linalg.vector_norm(weights, ord=2, dim=0)
    keep = torch.topk(importance, width).indices.sort().values
    with torch.no_grad():
        compact.encoder[0].weight.copy_(model.encoder[0].weight[keep])
        compact.encoder[0].bias.copy_(model.encoder[0].bias[keep])
        compact.encoder[1].weight.copy_(model.encoder[1].weight[keep])
        compact.encoder[1].bias.copy_(model.encoder[1].bias[keep])
        compact.encoder[3].weight.copy_(model.encoder[3].weight[:, keep])
        compact.encoder[3].bias.copy_(model.encoder[3].bias)
        compact.classifier.load_state_dict(model.classifier.state_dict())
    return compact


def train_student(student, loss_fn, train_data, seed, epochs, lr=5e-4):
    loader = DataLoader(
        train_data, batch_size=512, shuffle=True,
        generator=torch.Generator().manual_seed(seed)
    )
    optimizer = torch.optim.AdamW(student.parameters(), lr=lr, weight_decay=1e-4)
    best_f1, best_epoch, best_state = -1., 0, None
    for epoch in range(1, epochs + 1):
        student.train()
        losses = []
        for batch in loader:
            logits, embeddings = student(batch[0])
            loss = loss_fn(logits, embeddings, *batch[1:])
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(student.parameters(), 3.0)
            optimizer.step()
            losses.append(float(loss.item()))
        val_f1 = source_macro_f1(student)
        if val_f1 > best_f1:
            best_f1, best_epoch = val_f1, epoch
            best_state = copy.deepcopy(student.state_dict())
        print(f'Epoch {epoch}/{epochs}: loss={np.mean(losses):.4f}, '
              f'UNSW validation macro-F1={val_f1:.4f}')
    student.load_state_dict(best_state)
    return student.cpu().eval(), best_f1, best_epoch


source_train_ds = TensorDataset(
    torch.from_numpy(X_train), torch.from_numpy(y_train)
)


def pruning_loss(logits, embeddings, labels):
    return source_focal_loss(logits, labels)


models = {'Teacher_FP32': teacher}
source_rows = [{
    'Model': 'Teacher_FP32', 'Hidden_Width': 64,
    'Source_Validation_Macro_F1': source_macro_f1(teacher),
    'Encoder_Parameters': encoder_parameters(teacher), 'Best_Epoch': 0
}]

for width in (32, 16):
    print(f'\nPruning to {width} hidden units')
    torch.manual_seed(SEED + width)
    student = prune_encoder(teacher, width)
    student, score, epoch = train_student(
        student, pruning_loss, source_train_ds,
        seed=SEED + width, epochs=5
    )
    name = f'Pruned_{width}'
    models[name] = student
    torch.save({
        'state_dict': student.state_dict(), 'hidden_width': width,
        'n_classes': len(SOURCE_CLASSES), 'best_epoch': epoch,
        'source_validation_macro_f1': score, 'method': 'structured_pruning'
    }, OUT_DIR / f'{name.lower()}_checkpoint.pt')
    source_rows.append({
        'Model': name, 'Hidden_Width': width,
        'Source_Validation_Macro_F1': score,
        'Encoder_Parameters': encoder_parameters(student), 'Best_Epoch': epoch
    })

print('Pruning complete.')

## 4. Knowledge distillation

Distillation trains a smaller network from scratch. It learns from UNSW attack labels, the teacher's class probabilities, and its neural features. The loss weights are 40% supervised, 40% teacher probability matching, and 20% feature matching.

In [ ]:
@torch.inference_mode()
def teacher_signals(features, batch_size=2048):
    all_logits, all_features = [], []
    for start in range(0, len(features), batch_size):
        xb = torch.from_numpy(np.ascontiguousarray(features[start:start + batch_size]))
        logits, z = teacher(xb)
        all_logits.append(logits.numpy())
        all_features.append(z.numpy())
    return (np.vstack(all_logits).astype(np.float32),
            np.vstack(all_features).astype(np.float32))


t_logits, t_features = teacher_signals(X_train)
kd_dataset = TensorDataset(
    torch.from_numpy(X_train),
    torch.from_numpy(y_train),
    torch.from_numpy(t_logits),
    torch.from_numpy(t_features)
)


def distillation_loss(logits, z, labels, teacher_logits, teacher_z):
    temperature = 2.0
    supervised = source_focal_loss(logits, labels)
    soft_teacher = F.softmax(teacher_logits / temperature, dim=1)
    soft_student = F.log_softmax(logits / temperature, dim=1)
    soft_loss = F.kl_div(
        soft_student, soft_teacher, reduction='batchmean'
    ) * temperature ** 2
    feature_loss = F.smooth_l1_loss(z, teacher_z)
    return 0.40 * supervised + 0.40 * soft_loss + 0.20 * feature_loss


for width in (32, 16):
    print(f'\nDistillation to {width} hidden units')
    torch.manual_seed(44031 + width)
    student = CompactNIDS(width, len(SOURCE_CLASSES)).cpu()
    student, score, epoch = train_student(
        student, distillation_loss, kd_dataset,
        seed=44031 + width, epochs=8
    )
    name = f'Distilled_{width}'
    models[name] = student
    torch.save({
        'state_dict': student.state_dict(), 'hidden_width': width,
        'n_classes': len(SOURCE_CLASSES), 'best_epoch': epoch,
        'source_validation_macro_f1': score, 'method': 'knowledge_distillation',
        'temperature': 2.0,
        'loss_weights': {'supervised': 0.40, 'teacher_logits': 0.40, 'features': 0.20}
    }, OUT_DIR / f'{name.lower()}_checkpoint.pt')
    source_rows.append({
        'Model': name, 'Hidden_Width': width,
        'Source_Validation_Macro_F1': score,
        'Encoder_Parameters': encoder_parameters(student), 'Best_Epoch': epoch
    })

source_results = pd.DataFrame(source_rows)
source_results['Encoder_Parameter_Reduction_Pct'] = (
    100 * (1 - source_results['Encoder_Parameters'] / encoder_parameters(teacher))
)
source_results.to_csv(OUT_DIR / 'source_model_compression.csv', index=False)
display(source_results.round(5))

## 5. INT8 quantization and target evaluation

Dynamic INT8 quantization changes the teacher's linear layers without additional training, while keeping its embedding scaler and adapted classifier fixed. Pruned and distilled models each fit a classifier using the same 1,000 reviewed CIC examples. All models are compared on the CIC development partition.

In [ ]:
# Dynamic quantization applies to linear layers; the other layers stay in FP32.
with warnings.catch_warnings():
    warnings.filterwarnings('ignore', category=DeprecationWarning)
    int8_model = torch.ao.quantization.quantize_dynamic(
        copy.deepcopy(teacher), {nn.Linear}, dtype=torch.qint8
    ).cpu().eval()
models['Teacher_INT8'] = int8_model

reviewed_binary = pool_binary[selected]
reviewed_names = np.where(
    reviewed_binary == 0, 'Benign', pool_attack[selected]
)
attack_counts = pd.Series(reviewed_names[reviewed_binary == 1]).value_counts()
supported = set(attack_counts[attack_counts >= 3].index)
training_labels = np.array([
    name if name == 'Benign' or name in supported else 'Other_Attack'
    for name in reviewed_names
])
counts = pd.Series(training_labels).value_counts()
weights_by_name = {
    name: min(5.0, (counts.max() / count) ** 0.25)
    for name, count in counts.items()
}
sample_weights = np.array([weights_by_name[name] for name in training_labels])


def build_75_features(model):
    train_z = encode(model, X_train)
    scaler = RobustScaler(quantile_range=(10, 90)).fit(np.arcsinh(train_z))

    def combine(x):
        z = encode(model, x)
        scaled = np.clip(scaler.transform(np.arcsinh(z)), -8, 8)
        return np.column_stack((x, scaled)).astype(np.float32)

    return scaler, combine(X_pool), combine(X_dev)


adapters = {}
metrics = []
category_results = []

for name, model in models.items():
    print('Evaluating:', name)
    if name == 'Teacher_INT8':
        # Quantization changes the neural encoder, not the fitted classifier.
        _, scaler, classifier = adapters['Teacher_FP32']
        z = encode(model, X_dev)
        scaled = np.clip(scaler.transform(np.arcsinh(z)), -8, 8)
        dev75 = np.column_stack((X_dev, scaled)).astype(np.float32)
    else:
        scaler, pool75, dev75 = build_75_features(model)
        classifier = LogisticRegression(solver='lbfgs', max_iter=1200, random_state=13)
        classifier.fit(pool75[selected], training_labels, sample_weight=sample_weights)
    probs = classifier.predict_proba(dev75)
    predictions = classifier.classes_[probs.argmax(axis=1)]
    benign_idx = list(classifier.classes_).index('Benign')
    attack_scores = 1 - probs[:, benign_idx]
    attack_mask = dev_binary == 1

    result = {
        'Model': name,
        'Macro_F1': f1_score(
            dev_attack, predictions, labels=dev_categories,
            average='macro', zero_division=0
        ),
        'Binary_PR_AUC': average_precision_score(dev_binary, attack_scores),
        'Exact_Attack_Type_Accuracy': float(
            np.mean(predictions[attack_mask] == dev_attack[attack_mask])
        ),
        'Reviewed_Target_Labels': len(selected)
    }
    metrics.append(result)
    adapters[name] = (model, scaler, classifier)
    joblib.dump(scaler, OUT_DIR / f'{name.lower()}_embedding_scaler.joblib')
    joblib.dump(classifier, OUT_DIR / f'{name.lower()}_classifier.joblib')

    for category in dev_categories:
        mask = dev_attack == category
        category_results.append({
            'Model': name, 'Attack_Category': category,
            'Support': int(mask.sum()),
            'Recall': float(np.mean(predictions[mask] == category))
        })

evaluation = pd.DataFrame(metrics)
evaluation.to_csv(OUT_DIR / 'cic_development_comparison.csv', index=False)
pd.DataFrame(category_results).to_csv(
    OUT_DIR / 'cic_development_category_recall.csv', index=False
)
display(evaluation.round(5))

## 6. CPU inference and model size

The timing test includes feature processing, neural encoding, and classification. We measure batches of 1, 32, and 256 records. Storage measurements include the encoder, both fitted scalers, and the target classifier; they do not include the Python runtime.

In [ ]:
# Use the same input records for each model and batch size.
bench_data = target_adapt.reset_index(drop=True)
bench_rng = np.random.default_rng(44077)


def predict_raw_batch(frame, variant):
    encoder, embedding_scaler, classifier = variant
    features = transform_features(frame)
    z = encode(encoder, features)
    scaled_z = np.clip(embedding_scaler.transform(np.arcsinh(z)), -8, 8)
    combined = np.column_stack((features, scaled_z)).astype(np.float32)
    return classifier.predict_proba(combined)


# Every model processes the same prepared batches. The run order rotates.
latency_rows = []
for batch_size, repetitions in [(1, 30), (32, 25), (256, 20)]:
    batches = [
        bench_data.iloc[bench_rng.choice(
            len(bench_data), size=batch_size, replace=False
        )].copy()
        for _ in range(repetitions + 4)
    ]
    times = {name: [] for name in adapters}
    names = list(adapters)
    for i, batch in enumerate(batches):
        order = names[i % len(names):] + names[:i % len(names)]
        for name in order:
            start = time.perf_counter()
            _ = predict_raw_batch(batch, adapters[name])
            elapsed = (time.perf_counter() - start) * 1000
            if i >= 4:
                times[name].append(elapsed)
    for name in names:
        values = np.asarray(times[name])
        latency_rows.append({
            'Model': name, 'Batch_Size': batch_size,
            'Mean_Total_ms': float(values.mean()),
            'P95_Total_ms': float(np.percentile(values, 95)),
            'Median_Total_ms': float(np.median(values)),
            'Repetitions': repetitions
        })
    print('Finished batch size:', batch_size)

latency_df = pd.DataFrame(latency_rows)
latency_df.to_csv(OUT_DIR / 'cpu_inference_latency.csv', index=False)

size_rows = []
for name, (encoder_model, embedding_scaler, classifier) in adapters.items():
    state_path = OUT_DIR / f'{name.lower()}_encoder_state.pt'
    torch.save(encoder_model.encoder.state_dict(), state_path)
    component_paths = [
        state_path,
        OUT_DIR / f'{name.lower()}_embedding_scaler.joblib',
        OUT_DIR / f'{name.lower()}_classifier.joblib',
        SOURCE_SCALER
    ]
    size_rows.append({
        'Model': name,
        'Encoder_Bytes': state_path.stat().st_size,
        'Deployment_Artifact_Bytes': sum(p.stat().st_size for p in component_paths)
    })

size_df = pd.DataFrame(size_rows)
size_df['Deployment_Artifacts_KB'] = size_df['Deployment_Artifact_Bytes'] / 1024
size_df['Encoder_KB'] = size_df['Encoder_Bytes'] / 1024
size_df.to_csv(OUT_DIR / 'model_storage.csv', index=False)

comparison = (
    evaluation.merge(size_df, on='Model', validate='one_to_one')
    .merge(latency_df[latency_df['Batch_Size'] == 256], on='Model',
           validate='one_to_one')
)
teacher_size = float(comparison.loc[
    comparison['Model'] == 'Teacher_FP32', 'Deployment_Artifact_Bytes'
].iloc[0])
teacher_time = float(comparison.loc[
    comparison['Model'] == 'Teacher_FP32', 'Mean_Total_ms'
].iloc[0])
comparison['Artifact_Reduction_Pct'] = (
    100 * (1 - comparison['Deployment_Artifact_Bytes'] / teacher_size)
)
comparison['Batch256_Speedup'] = teacher_time / comparison['Mean_Total_ms']
comparison.to_csv(OUT_DIR / 'final_compression_comparison.csv', index=False)

print('Batch-256 development comparison')
display(comparison[
    ['Model', 'Macro_F1', 'Binary_PR_AUC', 'Deployment_Artifacts_KB',
     'Mean_Total_ms', 'P95_Total_ms', 'Batch256_Speedup']
].round(5))

## 7. Save experiment settings

The saved files include the compressed models, adapted classifiers, evaluation tables, and timing measurements. All reported compression results in this notebook come from the CIC development partition.

In [ ]:
settings = {
    'source_dataset': 'NF-UNSW-NB15-v2',
    'target_dataset': 'NF-CSE-CIC-IDS2018-v2',
    'source_checkpoint': checkpoint_path.name,
    'input_features': len(feature_names),
    'neural_features': 32,
    'target_classifier_features': 75,
    'reviewed_cic_labels': int(len(selected)),
    'target_selection_seed': 51,
    'pruning_widths': [32, 16],
    'pruning_finetune_epochs': 5,
    'distillation_widths': [32, 16],
    'distillation_epochs': 8,
    'distillation_temperature': 2.0,
    'distillation_loss_weights': [0.40, 0.40, 0.20],
    'int8_quantization': 'dynamic, linear layers',
    'development_records': len(target_dev),
    'benchmark_batch_sizes': [1, 32, 256],
    'benchmark_threads': 2,
    'evaluation': 'CIC development',
    'storage_units': 'bytes and KiB (1024 bytes)',
    'python_version': __import__('sys').version.split()[0],
    'pytorch_version': torch.__version__,
    'sklearn_version': __import__('sklearn').__version__,
}
with (OUT_DIR / 'compression_settings.json').open('w', encoding='utf-8') as f:
    json.dump(settings, f, indent=2)

print('Compression experiment complete.')
print('Results:', OUT_DIR)
print('Models compared:', ', '.join(adapters))
print('Evaluation set: CIC development')